# Reinforcement Learning Trading Agent

This notebook trains an RL agent that learns optimal buy/sell decisions by maximizing profit,
rather than fitting to manually-defined labels.

## Key Advantages over Classification Approach:
- **No label bias**: The agent discovers optimal trading patterns itself
- **Profit-focused**: Directly optimizes for profit, not classification accuracy
- **Adaptive**: Learns to handle different market conditions

## Algorithm Options:
- **PPO** (Recommended): Stable, sample-efficient, works well for trading
- **A2C**: Faster but potentially less stable
- **DQN**: Value-based, good for discrete actions

**IMPORTANT**: This notebook requires `gymnasium` and `stable-baselines3` packages.

## GitHub Repository Setup

In [ ]:
# GitHub Configuration
GITHUB_USERNAME = "tr4m0ryp"  # Replace with your GitHub username
GITHUB_TOKEN = ""  # Replace with your GitHub Personal Access Token
GITHUB_REPO_URL = "https://github.com/tr4m0ryp/GMGN_TradingBot.git"

print("GitHub credentials configured")
print(f"Username: {GITHUB_USERNAME}")
print(f"Repository: {GITHUB_REPO_URL}")

In [ ]:
import os
import subprocess

os.chdir('/content')

repo_name = GITHUB_REPO_URL.split('/')[-1].replace('.git', '')
repo_path = f'/content/{repo_name}'

if os.path.exists(repo_path):
    print(f"Repository '{repo_name}' already exists at {repo_path}")
    print("Skipping clone. Use git pull to update.")
else:
    clone_url = GITHUB_REPO_URL.replace('https://', f'https://{GITHUB_USERNAME}:{GITHUB_TOKEN}@')
    result = subprocess.run(['git', 'clone', clone_url], capture_output=True, text=True, check=True)
    print(f"Successfully cloned repository to {repo_path}")

import sys
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

os.chdir(repo_path)
print(f"Current directory: {os.getcwd()}")

## Install RL Dependencies

In [ ]:
# Install required packages for RL
!pip install gymnasium stable-baselines3[extra] tensorboard -q

print("RL dependencies installed successfully")

## Check GPU

In [ ]:
import torch

print("=" * 50)
print("GPU STATUS")
print("=" * 50)

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")
    total_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"Total Memory: {total_memory:.2f} GB")
    device = 'cuda'
else:
    print("No GPU available, using CPU")
    device = 'cpu'

print("=" * 50)

## Import Modules

In [ ]:
import os
import sys
import json
import numpy as np

repo_name = GITHUB_REPO_URL.split('/')[-1].replace('.git', '')
src_path = f'/content/{repo_name}/ai_model/src'
ai_model_path = f'/content/{repo_name}/ai_model'

if src_path not in sys.path:
    sys.path.insert(0, src_path)

os.chdir(ai_model_path)
print(f"Working directory: {os.getcwd()}")

try:
    from rl import TradingEnvironment, RLTradingAgent, train_rl_agent
    from rl.environment import MultiTokenTradingEnvironment
    from rl.trainer import load_token_candles, backtest_rl_agent
    from data.preparation import parse_candles
    
    print("\nAll RL modules imported successfully!")
    print("\nAvailable components:")
    print("  - TradingEnvironment: Gym-compatible trading env")
    print("  - RLTradingAgent: PPO/A2C/DQN agent wrapper")
    print("  - train_rl_agent: Complete training pipeline")
    
except ImportError as e:
    print(f"Import failed: {e}")
    print("Make sure the rl module exists in ai_model/src/")

## Configuration

In [ ]:
# ============================================================
# RL TRAINING CONFIGURATION
# ============================================================

# Algorithm: 'ppo' (recommended), 'a2c', or 'dqn'
ALGORITHM = 'ppo'

# Training parameters
TOTAL_TIMESTEPS = 500_000     # Total training steps
LEARNING_RATE = 3e-4          # Learning rate
N_ENVS = 4                    # Parallel environments

# Checkpointing
EVAL_FREQ = 10_000            # Evaluation frequency
SAVE_FREQ = 50_000            # Checkpoint frequency

# ============================================================

print("Training Configuration:")
print(f"  Algorithm: {ALGORITHM.upper()}")
print(f"  Total Timesteps: {TOTAL_TIMESTEPS:,}")
print(f"  Learning Rate: {LEARNING_RATE}")
print(f"  Parallel Environments: {N_ENVS}")
print(f"  Evaluation Frequency: {EVAL_FREQ:,}")
print(f"  Checkpoint Frequency: {SAVE_FREQ:,}")

## Load and Explore Data

In [ ]:
import pandas as pd

repo_name = GITHUB_REPO_URL.split('/')[-1].replace('.git', '')
data_dir = f'/content/{repo_name}/ai_model/data'
raw_csv = f'{data_dir}/raw/rawdata.csv'

print(f"Loading data from: {raw_csv}")

# Load raw data
df = pd.read_csv(
    raw_csv,
    quotechar='"',
    escapechar='\\',
    on_bad_lines='warn',
    engine='python'
)

print(f"\nLoaded {len(df)} tokens")
print(f"Columns: {list(df.columns)}")

# Parse candles for all tokens
all_candles = []
for idx in range(len(df)):
    try:
        candles = parse_candles(df.iloc[idx]['candles'])
        if len(candles) >= 50:
            all_candles.append(candles)
    except Exception:
        continue

print(f"\nValid tokens with >= 50 candles: {len(all_candles)}")
print(f"Average candles per token: {np.mean([len(c) for c in all_candles]):.0f}")
print(f"Total candles: {sum(len(c) for c in all_candles):,}")

## Quick Test: Single Token Environment

In [ ]:
# Test the environment with a single token
test_candles = all_candles[0]
print(f"Testing environment with token 0 ({len(test_candles)} candles)")

env = TradingEnvironment(test_candles)
obs, info = env.reset()

print(f"\nObservation shape: {obs.shape}")
print(f"Action space: {env.action_space}")
print(f"  0 = HOLD")
print(f"  1 = BUY")
print(f"  2 = SELL")

# Run random policy for a few steps
print("\nRunning random policy for 100 steps...")
total_reward = 0
for _ in range(100):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    if terminated or truncated:
        break

print(f"\nRandom policy results:")
print(f"  Total Reward: {total_reward:.4f}")
print(f"  PnL: {info['total_pnl']:.6f} SOL")
print(f"  Trades: {info['n_trades']}")
print(f"  Win Rate: {info['win_rate']:.2%}")

env.close()

## Train RL Agent

In [ ]:
# Create output directory
repo_name = GITHUB_REPO_URL.split('/')[-1].replace('.git', '')
output_dir = f'/content/{repo_name}/ai_model/models/rl'
os.makedirs(output_dir, exist_ok=True)

print(f"Output directory: {output_dir}")
print("\nStarting RL training...")
print("="*60)

# Train the agent
results = train_rl_agent(
    data_dir=data_dir,
    output_dir=output_dir,
    algorithm=ALGORITHM,
    total_timesteps=TOTAL_TIMESTEPS,
    learning_rate=LEARNING_RATE,
    n_envs=N_ENVS,
    eval_freq=EVAL_FREQ,
    save_freq=SAVE_FREQ,
    device=device,
    verbose=1,
)

print("\n" + "="*60)
print("TRAINING COMPLETE")
print("="*60)

## Visualize Training Progress

In [ ]:
# Load TensorBoard for training visualization
%load_ext tensorboard

repo_name = GITHUB_REPO_URL.split('/')[-1].replace('.git', '')
log_dir = f'/content/{repo_name}/ai_model/models/rl/logs'

%tensorboard --logdir {log_dir}

## Evaluate Trained Agent

In [ ]:
from stable_baselines3 import PPO, A2C, DQN

# Load the best model
repo_name = GITHUB_REPO_URL.split('/')[-1].replace('.git', '')
model_path = f'/content/{repo_name}/ai_model/models/rl/best_model.zip'

print(f"Loading model from: {model_path}")

# Create evaluation environment
eval_candles = all_candles[-10:]  # Last 10 tokens for evaluation
eval_env = MultiTokenTradingEnvironment(eval_candles)

# Load model (use the algorithm you trained with)
if ALGORITHM == 'ppo':
    model = PPO.load(model_path, env=eval_env)
elif ALGORITHM == 'a2c':
    model = A2C.load(model_path, env=eval_env)
else:
    model = DQN.load(model_path, env=eval_env)

print("Model loaded successfully")

# Evaluate on multiple episodes
print("\nEvaluating on 20 episodes...")

episode_pnls = []
episode_trades = []
episode_win_rates = []

for ep in range(20):
    obs, _ = eval_env.reset()
    done = False
    
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = eval_env.step(action)
        done = terminated or truncated
    
    episode_pnls.append(info['total_pnl'])
    episode_trades.append(info['n_trades'])
    episode_win_rates.append(info['win_rate'])

print("\n" + "="*60)
print("EVALUATION RESULTS")
print("="*60)
print(f"Mean PnL: {np.mean(episode_pnls):.6f} +/- {np.std(episode_pnls):.6f} SOL")
print(f"Total PnL: {sum(episode_pnls):.6f} SOL")
print(f"Mean Trades: {np.mean(episode_trades):.1f}")
print(f"Mean Win Rate: {np.mean(episode_win_rates):.2%}")
print(f"Best Episode PnL: {max(episode_pnls):.6f} SOL")
print(f"Worst Episode PnL: {min(episode_pnls):.6f} SOL")
print("="*60)

eval_env.close()

## Detailed Backtest

In [ ]:
# Run detailed backtest on a single token
test_token_idx = 5  # Choose a token to backtest
test_candles = all_candles[test_token_idx]

print(f"Backtesting on token {test_token_idx} ({len(test_candles)} candles)")

backtest_results = backtest_rl_agent(
    model_path=model_path,
    candles=test_candles,
    verbose=True,
)

# Plot action distribution
import matplotlib.pyplot as plt

action_dist = backtest_results['action_distribution']
actions = ['HOLD', 'BUY', 'SELL']
counts = [action_dist.get(i, 0) for i in range(3)]

plt.figure(figsize=(8, 4))
plt.bar(actions, counts, color=['gray', 'green', 'red'])
plt.title('Action Distribution During Backtest')
plt.ylabel('Count')
plt.xlabel('Action')
for i, c in enumerate(counts):
    plt.text(i, c + 5, str(c), ha='center')
plt.tight_layout()
plt.show()

## Save Results

In [ ]:
from google.colab import files

# Download the trained model and results
repo_name = GITHUB_REPO_URL.split('/')[-1].replace('.git', '')

# Download final model
model_file = f'/content/{repo_name}/ai_model/models/rl/final_model.zip'
if os.path.exists(model_file):
    files.download(model_file)
    print(f"Downloaded: {model_file}")

# Download results
results_file = f'/content/{repo_name}/ai_model/models/rl/training_results.json'
if os.path.exists(results_file):
    files.download(results_file)
    print(f"Downloaded: {results_file}")

print("\nTraining complete! Download your model and results above.")